# staged_75 공간 분포 토큰 선택: 5,000장 예비 실험

ImageNet-1K validation의 동일 표본에서 기존 CLS attention Top-K와 4×4 공간 분포 선택을 비교합니다. 학습 없이 T4에서 실행한 결과를 아래 출력에 보존했습니다. 자세한 방법과 해석은 aseline/reports/2026-11-17/staged75-spatial-pilot/README.md를 참고하세요.


In [ ]:
import os, sys, subprocess, json, shutil
from pathlib import Path

os.environ['MPLBACKEND'] = 'Agg'
PROJECT = Path('/content/OnDevice')
BRANCH = 'codex/layer-wise-drop'
OFFICIAL_COMMIT = '9254c4950d3159937ad682592d58a2be69708c42'

def run(*command, **kwargs):
    command = [str(x) for x in command]
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, **kwargs) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
        if code:
            raise subprocess.CalledProcessError(code, command)

if not PROJECT.exists():
    run('git', 'clone', '--branch', BRANCH, 'https://github.com/hee4040/OnDevice.git', PROJECT)
os.chdir(PROJECT)
print('Project revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
# Separate Python 3.10 avoids changing the Colab kernel or its preinstalled packages.
run(sys.executable, '-m', 'pip', 'install', 'uv==0.9.8')
PYTHON = Path('/content/ondevice-env/bin/python')
if not PYTHON.exists():
    run(sys.executable, '-m', 'uv', 'venv', '--python', '3.10', '/content/ondevice-env')
run(sys.executable, '-m', 'uv', 'pip', 'install', '--python', PYTHON, '-r', 'requirements-colab.txt')
official = PROJECT / 'third_party/ThinkingViT'
if not official.exists():
    run('git', 'clone', 'https://github.com/ds-kiel/ThinkingViT.git', official)
run('git', '-C', official, 'checkout', OFFICIAL_COMMIT)
run(PYTHON, '-c', 'import torch; print(torch.__version__, torch.cuda.get_device_name() if torch.cuda.is_available() else "NO GPU"); assert torch.cuda.is_available(), "Select GPU runtime first"')
run(PYTHON, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_layerwise_drop.py', '-v')
run(PYTHON, 'scripts/prepare_weights.py')
run(PYTHON, 'baseline/check_layerwise_model.py')


Cloning into '/content/OnDevice'...
Project revision: 8bd600696223b7863302b243467b754c4246274c
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 21.8 MB/s  0:00:01
Using CPython 3.10.19
Creating virtual environment at: /content/ondevice-env
Activate with: source /content/ondevice-env/bin/activate
Using Python 3.10.19 environment at: /content/ondevice-env
Resolved 46 packages in 490ms
Prepared 46 packages in 1m 03s
Installed 46 packages in 418ms
 + absl-py==2.2.2
 + certifi==2026.10.23
 + charset-normalizer==3.5.2
 + contourpy==1.3.2
 + cycler==0.12.1
 + filelock==4.1.0
 + fonttools==4.65.0
 + fsspec==2026.9.0
 + huggingface-hub==0.26.5
 + idna==3.20
 + jinja2==3.1.6
 + kiwisolver==1.5.1
 + markupsafe==3.0.4
 + matplotlib==3.10.7
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + nvidia-cublas-cu12==12.4.5.8
 + nvidia-cuda-cupti-cu12==12.4.127
 + nvidia-cuda-nvrtc-cu12==12.4.127
 + nvidia-cuda-runtime-cu12==12.4.127
 + nvidia-cudnn-cu12==9.1.0.70
 + nvidia-cufft-cu12==11.2.1.

## 기존 Drive 데이터 연결

MyDrive/OnDevice/data/imagenet1k/validation의 기존 검증 이미지를 직접 읽습니다. 전체 50,000장을 Colab 디스크로 다시 복사하지 않습니다.


In [4]:
from google.colab import drive
drive.mount('/content/drive')
STORE = Path('/content/drive/MyDrive/OnDevice')
DATA_DRIVE = STORE / 'data/imagenet1k'
RESULTS = STORE / 'results/layerwise-drop-v1'
RESULTS.mkdir(parents=True, exist_ok=True)
shutil.copy2(PROJECT / 'results/layerwise-model-check.json', RESULTS / 'model-check.json')
print('Persistent data:', DATA_DRIVE)
print('Persistent results:', RESULTS)
print('Drive free GiB:', round(shutil.disk_usage(STORE).free / 1024**3, 1))


Mounted at /content/drive
Persistent data: /content/drive/MyDrive/OnDevice/data/imagenet1k
Persistent results: /content/drive/MyDrive/OnDevice/results/layerwise-drop-v1
Drive free GiB: 101.6


In [5]:
# Paired 5,000-image pilot: original staged_75 vs spatially balanced staged_75.
PILOT = r'''
import json, math, sys, time
from pathlib import Path
import torch
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder
PROJECT=Path('/content/OnDevice')
sys.path.insert(0,str(PROJECT/'baseline'))
from evaluate_drop import load_model
from evaluate_h1_diagnosis import wide_input
from selective_refinement import forward_narrow_with_attention
from layerwise_drop import layerwise_logits, token_counts, wide_macs
torch.manual_seed(42); torch.set_num_threads(1)
torch.backends.cuda.matmul.allow_tf32=True
torch.backends.cudnn.benchmark=True
model=load_model(PROJECT/'third_party/ThinkingViT', PROJECT/'checkpoints/thinkingvit_3h6h_ema.pth', device='cuda')
from timm.data import create_transform, resolve_data_config
ratios=[1.0]*4+[0.875]*4+[0.75]*4
assert token_counts(ratios)==(197,)*4+(173,)*4+(148,)*4
grid=((torch.arange(14,device='cuda')[:,None]*4//14)*4+(torch.arange(14,device='cuda')[None,:]*4//14)).reshape(-1)
def spatial_scores(scores):
    p=scores[:,1:]
    forced=torch.zeros_like(p,dtype=torch.bool)
    for region in range(16):
        winners=p.masked_fill(grid[None,:]!=region,-1).topk(6,dim=1).indices
        forced.scatter_(1,winners,True)
    boosted=torch.where(forced,p.max(dim=1,keepdim=True).values+1,p)
    return torch.cat((scores[:,:1],boosted),dim=1)
root=Path('/content/drive/MyDrive/OnDevice/data/imagenet1k/validation')
assert root.is_dir(),f'Existing Drive validation data missing: {root}'
config=resolve_data_config({},model=model,use_test_size=True)
dataset=ImageFolder(root,transform=create_transform(**config,is_training=False))
assert len(dataset)==50000
indices=torch.randperm(len(dataset),generator=torch.Generator().manual_seed(42))[:5000].tolist()
loader=DataLoader(Subset(dataset,indices),batch_size=32,num_workers=2,pin_memory=True,shuffle=False)
base_flags=[]; spatial_flags=[]; routed_flags=[]; entropies=[]; changed=[]
t0=time.time()
with torch.inference_mode():
    for b,(images,targets) in enumerate(loader):
        images,targets=images.cuda(non_blocking=True),targets.cuda(non_blocking=True)
        features,narrow,scores=forward_narrow_with_attention(model,images)
        entropy=model.entropy(narrow)[:,0]
        routed=entropy>=1.0
        old_ok=narrow.argmax(1)==targets
        new_ok=old_ok.clone()
        changes=torch.zeros(len(images),device='cuda',dtype=torch.int32)
        if routed.any():
            x0=wide_input(model,images[routed],features[routed])
            raw=scores[routed]; balanced=spatial_scores(raw)
            old_logits=layerwise_logits(model,x0,raw,ratios)
            new_logits=layerwise_logits(model,x0,balanced,ratios)
            old_ok[routed]=old_logits.argmax(1)==targets[routed]
            new_ok[routed]=new_logits.argmax(1)==targets[routed]
            a=torch.zeros_like(raw[:,1:],dtype=torch.bool).scatter_(1,raw[:,1:].topk(147,dim=1).indices,True)
            c=torch.zeros_like(a).scatter_(1,balanced[:,1:].topk(147,dim=1).indices,True)
            changes[routed]=((a^c).sum(1)//2).int()
        base_flags.append(old_ok.cpu()); spatial_flags.append(new_ok.cpu())
        routed_flags.append(routed.cpu()); entropies.append(entropy.cpu()); changed.append(changes.cpu())
        if (b+1)%10==0 or b+1==len(loader):
            print(f'{b+1}/{len(loader)} batches, elapsed {time.time()-t0:.0f}s',flush=True)
old=torch.cat(base_flags); new=torch.cat(spatial_flags)
routed=torch.cat(routed_flags); changes=torch.cat(changed)
d=new.double()-old.double()
se=d.std(unbiased=True).item()/math.sqrt(len(d))
summary=dict(sample_size=len(d),routed=int(routed.sum()),seed=42,threshold=1.0,
 policy='staged_75, 4x4 regions, top 6 attention patches per region then global Top-K',
 original_top1=100*old.double().mean().item(),
 spatial_top1=100*new.double().mean().item(),
 difference_pp=100*d.mean().item(),
 ci95_pp=[100*(d.mean().item()+sign*1.96*se) for sign in (-1,1)],
 spatial_only_correct=int(((~old)&new).sum()),
 original_only_correct=int((old&(~new)).sum()),
 mean_replaced_patches_on_routed=changes[routed].double().mean().item(),
 images_with_changed_patch_set=int((changes>0).sum()),
 same_analytic_wide_gmacs=wide_macs(token_counts(ratios))/1e9,
 note='Small paired pilot on previously used validation; interval descriptive, not independent confirmation.')
out=Path('/content/drive/MyDrive/OnDevice/results/layerwise-spatial-pilot')
out.mkdir(parents=True,exist_ok=True)
(out/'summary.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
torch.save(dict(indices=torch.tensor(indices),entropy=torch.cat(entropies),routed=routed,
 original_correct=old,spatial_correct=new,replaced_patches=changes),out/'per_image.pt')
print(json.dumps(summary,indent=2,ensure_ascii=False),flush=True)
print('Saved to',out,flush=True)
'''
run(PYTHON,'-u','-c',PILOT)


10/157 batches, elapsed 74s
20/157 batches, elapsed 145s
30/157 batches, elapsed 216s
40/157 batches, elapsed 285s
50/157 batches, elapsed 353s
60/157 batches, elapsed 421s
70/157 batches, elapsed 491s
80/157 batches, elapsed 563s
90/157 batches, elapsed 638s
100/157 batches, elapsed 716s
110/157 batches, elapsed 788s
120/157 batches, elapsed 857s
130/157 batches, elapsed 924s
140/157 batches, elapsed 994s
150/157 batches, elapsed 1062s
157/157 batches, elapsed 1105s
{
  "sample_size": 5000,
  "routed": 1747,
  "seed": 42,
  "threshold": 1.0,
  "policy": "staged_75, 4x4 regions, top 6 attention patches per region then global Top-K",
  "original_top1": 80.96,
  "spatial_top1": 80.92,
  "difference_pp": -0.04,
  "ci95_pp": [
    -0.16396872322028516,
    0.08396872322028517
  ],
  "spatial_only_correct": 4,
  "original_only_correct": 6,
  "mean_replaced_patches_on_routed": 2.7406983400114484,
  "images_with_changed_patch_set": 1502,
  "same_analytic_wide_gmacs": 4.017509376,
  "note": "S